### Instalação das dependências

Os pacotes abaixo são necessários para treinar, avaliar e comparar os modelos Champion e Challenger. Execute esta célula antes de qualquer outra.

In [ ]:
!pip install scikit-learn pandas joblib matplotlib -q

# Código 3.1 — Esteira de Qualidade de Modelos (Model Gate)

**Objetivo:** Construir um portal automatizado de decisão (*Quality Gate*) que avalia um novo modelo candidato (*Challenger*) contra o modelo atual em produção (*Champion*) em um conjunto de dados retido (*Holdout*).

### Regras de Aprovação do Model Gate

<table style="width: 100%; border-collapse: collapse; text-align: center; font-family: sans-serif; font-size: 13px; margin: 10px 0;">
  <tr style="background-color: #f8fafc;">
    <td style="padding: 6px; border: 1px solid #e2e8f0; font-weight: bold; color: #334155; width: 20%;">Challenger</td>
    <td style="padding: 6px; border: 1px solid #e2e8f0; color: #94a3b8; width: 4%; font-weight: bold;">&rarr;</td>
    <td style="padding: 6px; border: 1px solid #e2e8f0; background-color: #eff6ff; font-weight: bold; color: #1e40af; width: 24%;">Teste 1: AUC &ge; 0.82</td>
    <td style="padding: 6px; border: 1px solid #e2e8f0; color: #94a3b8; width: 4%; font-weight: bold;">&rarr;</td>
    <td style="padding: 6px; border: 1px solid #e2e8f0; background-color: #eff6ff; font-weight: bold; color: #1e40af; width: 24%;">Teste 2: F1 &ge; 0.70</td>
    <td style="padding: 6px; border: 1px solid #e2e8f0; color: #94a3b8; width: 4%; font-weight: bold;">&rarr;</td>
    <td style="padding: 6px; border: 1px solid #e2e8f0; background-color: #f0fdf4; font-weight: bold; color: #166534; width: 20%;">Promoção<br><span style="font-size: 11px; font-weight: normal; color: #16a34a;">(Champion +0.5%)</span></td>
  </tr>
</table>

### Passo 1: Preparação do Ambiente e Geração de Artefatos Sintéticos

Caso os arquivos de validação não existam, simulamos o treinamento do modelo **Champion** (versão legada mais simples) e do modelo **Challenger** (novo candidato a promoção).

In [ ]:
from pathlib import Path
import joblib
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

HOLDOUT_PATH = Path("data/holdout.csv")
CHAMPION_PATH = Path("models/champion.joblib")
CHALLENGER_PATH = Path("models/challenger.joblib")

if not (HOLDOUT_PATH.exists() and CHAMPION_PATH.exists() and CHALLENGER_PATH.exists()):
    Path("data").mkdir(exist_ok=True)
    Path("models").mkdir(exist_ok=True)

    # 1. Gerar dados sintéticos
    X, y = make_classification(
        n_samples=1000, n_features=10, n_informative=6, random_state=42, weights=[0.7, 0.3]
    )
    df = pd.DataFrame(X, columns=[f"f{i}" for i in range(10)])
    df["alvo"] = y

    train_df, holdout_df = train_test_split(df, test_size=0.3, random_state=42, stratify=y)
    X_train = train_df.drop(columns=["alvo"])
    y_train = train_df["alvo"]

    # 2. Treinar Champion (modelo legado mais fraco)
    champion_model = Pipeline([
        ("scaler", StandardScaler()),
        ("model", GradientBoostingClassifier(n_estimators=10, max_depth=1, random_state=42))
    ])
    champion_model.fit(X_train, y_train)

    # 3. Treinar Challenger (novo modelo com maior capacidade)
    challenger_model = Pipeline([
        ("scaler", StandardScaler()),
        ("model", RandomForestClassifier(n_estimators=150, random_state=42, n_jobs=-1))
    ])
    challenger_model.fit(X_train, y_train)

    # 4. Salvar artefatos
    holdout_df.to_csv(HOLDOUT_PATH, index=False)
    joblib.dump(champion_model, CHAMPION_PATH)
    joblib.dump(challenger_model, CHALLENGER_PATH)
    print("Arquivos de Holdout e modelos Champion/Challenger criados com sucesso!")
else:
    print("Artefatos encontrados em disco.")

### Passo 2: Carga do Dataset Holdout e dos Modelos Salvos

Carregamos o dataset de validação e os artefatos serializados.

**Entendendo as variáveis de preparação e simulação:**
- `HOLDOUT_PATH`: Caminho do arquivo CSV contendo o conjunto de teste isolado (*Holdout*), reservado exclusivamente para validação das regras do gate.
- `CHAMPION_PATH`: Caminho onde o modelo ativo em produção (*Champion*) fica armazenado.
- `CHALLENGER_PATH`: Caminho onde a nova versão candidata a substituir a produção (*Challenger*) fica armazenada.
- `champion_model`: Modelo treinado com uma arquitetura deliberadamente mais simples (`GradientBoostingClassifier` com poucas árvores) para simular o modelo legado.
- `challenger_model`: Novo modelo treinado com maior capacidade (`RandomForestClassifier` com 150 árvores) para simular a proposta de melhoria.

In [ ]:
import joblib
import pandas as pd

# Carregar Holdout
holdout = pd.read_csv(HOLDOUT_PATH)
X_holdout = holdout.drop(columns=["alvo"])
y_holdout = holdout["alvo"]

# Carregar modelos
champion = joblib.load(CHAMPION_PATH)
challenger = joblib.load(CHALLENGER_PATH)

print(f"Dataset Holdout carregado com {X_holdout.shape[0]} amostras.")

### Passo 3: Avaliação de Desempenho e Matriz de Confusão Comparativa

Exibimos as métricas técnicas e a matriz de confusão visual do modelo Challenger.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, f1_score, roc_auc_score

def avaliar_modelo(modelo, X, y) -> dict[str, float]:
    probabilidades = modelo.predict_proba(X)[:, 1]
    previsoes = modelo.predict(X)
    return {
        "auc": roc_auc_score(y, probabilidades),
        "f1": f1_score(y, previsoes),
    }

metricas_champion = avaliar_modelo(champion, X_holdout, y_holdout)
metricas_challenger = avaliar_modelo(challenger, X_holdout, y_holdout)

print("=== DESEMPENHO NO HOLDOUT ===")
print(f"Champion   : AUC = {metricas_champion['auc']:.4f} | F1 = {metricas_champion['f1']:.4f}")
print(f"Challenger : AUC = {metricas_challenger['auc']:.4f} | F1 = {metricas_challenger['f1']:.4f}")

# Matriz de Confusão Visual do Challenger
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_estimator(challenger, X_holdout, y_holdout, ax=ax, cmap="Blues")
plt.title("Matriz de Confusão — Modelo Challenger")
plt.tight_layout()
plt.show()

### Passo 4: Validação das Regras do Model Gate

Aplicamos os critérios rígidos de aprovação do gate de qualidade.

In [ ]:
# Definição das regras rígidas do Gate
auc_minima = 0.82
f1_minimo = 0.70
margem_minima_ganho = 0.005

# 1. Regra Absoluta de AUC
if metricas_challenger["auc"] < auc_minima:
    raise SystemExit(f"Gate Reprovado: AUC do Challenger ({metricas_challenger['auc']:.4f}) abaixo do mínimo ({auc_minima})")

# 2. Regra Absoluta de F1-Score
if metricas_challenger["f1"] < f1_minimo:
    raise SystemExit(f"Gate Reprovado: F1 do Challenger ({metricas_challenger['f1']:.4f}) abaixo do mínimo ({f1_minimo})")

# 3. Regra Relativa (Superar o Champion)
auc_necessaria = metricas_champion["auc"] + margem_minima_ganho
if metricas_challenger["auc"] < metricas_champion["auc"] + margem_minima_ganho:
    raise SystemExit(f"Gate Reprovado: Challenger não superou a margem necessária em relação ao Champion ({auc_necessaria:.4f})")

print("GATE APROVADO! O modelo Challenger foi promovido e está elegível para implantação em produção.")

**Entendendo as variáveis e regras de decisão do Model Gate:**
- `auc_minima = 0.82`: Nota de corte absoluta para a área sob a curva ROC. Se o candidato não atingir esse valor, é reprovado automaticamente.
- `f1_minimo = 0.70`: Nota de corte absoluta para a métrica F1 (equilíbrio entre precisão e revocação). Evita aprovar modelos com alto ROC-AUC mas péssimo limiar de decisão.
- `margem_minima_ganho = 0.005`: Ganho mínimo relativo exigido (0,5% de AUC) acima do modelo *Champion*. Impede a substituição de modelos em produção por ganhos irrelevantes que não justificam o risco de re-implantação.
- `auc_necessaria`: Soma do desempenho do Champion atual com a margem mínima de ganho necessária para aprovação.

---

## Conclusão da Aula — Model Gate

O **Model Gate** atua como uma barreira automatizada de qualidade. Ele impede que novos modelos com métricas inferiores ou com ganho marginal insuficiente cheguem à produção, protegendo a operação de degradação.